In [2]:
import json
import pandas as pd
from pathlib import Path

In [27]:
def read_all():
    with open(DATA_DIR / "corpus.jsonl", "r", encoding="utf-8") as f:
        corpus = [json.loads(line) for line in f]


    with open(DATA_DIR / "queries.jsonl", "r", encoding="utf-8") as f:
        queries = [json.loads(line) for line in f]

    with open(DATA_DIR / "test_queries.jsonl", "r", encoding="utf-8") as f:
        test_queries = [json.loads(line) for line in f]

    train_q = pd.read_csv(DATA_DIR / "qrels/train.tsv", sep="\t")
    dev_q = pd.read_csv(DATA_DIR / "qrels/dev.tsv", sep="\t")
    test_q = pd.read_csv(DATA_DIR / "qrels/test.tsv", sep="\t")

    return corpus, queries, test_queries, train_q, dev_q, test_q



# FEVER

In [11]:
dataset = 'fever'
DATA_DIR = Path(f"../../{dataset}")

In [12]:
corpus, queries, test_queries, train_q, dev_q, test_q = read_all()

FileNotFoundError: [Errno 2] No such file or directory: 'hotpotqa\\corpus.jsonl'

In [ ]:
print(f"dataset -- {dataset} --")
print(f"Numero documenti del corpus: {len(corpus):,}")
print(f"Numero query:                {len(queries):,}")
print(f"Numero query di test:        {len(test_queries):,}")

dataset -- fever --
Numero documenti del corpus: 5,416,568
Numero query:               123,142
Numero query di test:      6,666


## Analizziamo i file

In [14]:
queries[20]

{'_id': '15812',
 'text': 'Peggy Sue Got Married is a Egyptian film released in 1986.',
 'metadata': {'verifiable': 'VERIFIABLE',
  'label': 'REFUTES',
  'evidence': [[[31205, 37902, 'Peggy_Sue_Got_Married', 0],
    [31205, 37902, 'Francis_Ford_Coppola', 0]],
   [[31211, 37908, 'Peggy_Sue_Got_Married', 0]]]}}

Una query ha i seguenti campi:
* `_id`: identificatore univoco
* `text`: e' un'asserzione, tecnicamente si chiama **claim** che il sistema dovra' verificare
* `label`: tra i metadata e' quello piu' interessante. Sarebbe la verita' del claim, l'esito della ricerca. Nell'esempio sopra l'affermazione e' falsa (**REFUTES**) 
* `evidence`: qui, a parte gli indici che il sistema usera' per determinare la verita' dei claim, abbiamo dei valori str, come in questo caso **'Peggy_Sue_Got_Married'** che sono chiave esterna per agganciarsi ai documenti del corpus

In [15]:
print(next(
    d for d in corpus
    if d["_id"] == "Peggy_Sue_Got_Married"
))

print(next(
    d for d in corpus
    if d["_id"] == "Francis_Ford_Coppola"
))

{'_id': 'Peggy_Sue_Got_Married', 'title': 'Peggy Sue Got Married', 'text': 'Peggy Sue Got Married is a 1986 American comedy-drama film directed by Francis Ford Coppola starring Kathleen Turner as a woman on the verge of a divorce , who finds herself transported back to the days of her senior year in high school in 1960 . The film was written by husband and wife team Jerry Leichtling and Arlene Sarner .   The film was nominated for three Academy Awards : Best Actress ( Turner ) , Best Cinematography , and Best Costume Design . In addition , Turner was nominated for Best Foreign Actress at the Sant Jordi Awards .', 'metadata': {}}
{'_id': 'Francis_Ford_Coppola', 'title': 'Francis Ford Coppola', 'text': "Francis Ford Coppola ( -LSB- pronˈkoʊpələ -RSB- born April 7 , 1939 ) , also credited as Francis Coppola , is a semi-retired American film director , producer , and screenwriter . He is considered to have been a central figure of the New Hollywood wave of filmmaking .   After directing Th

Nei documenti del corpus abbiamo:
* `_id`: chiave primaria costituita da un testo, simil titolo, che connette con le chiavi esterne (*evidence*) delle query
* `title`: titolo univoco del documento
* `text`: il testo del documento


In [16]:
train_q

,query-id,corpus-id,score
0,75397,Fox_Broadcasting_Company,1
1,75397,Nikolaj_Coster-Waldau,1
2,150448,Roman_Atwood,1
3,214861,History_of_art,1
4,156709,Adrienne_Bailon,1
...,...,...,...
140080,149256,Taal_(film),1
140081,13287,Rooney_Mara,1
140082,13287,Her_(film),1
140083,13114,Gimli_(Middle-earth),1


In `/qrels` abbiamo 3 file, gia splittati in train, val e test<br>.
Rappresentano i "ponti" che legano le query ai documenti del corpus. In pratica:
* `query-id` e' la chiave che connette agli id delle query
* `corpus-id` e' la chiave che connette ai docuemnti del corpus
* `score` e' il metadato, sempre 1, che indica che il documento (corpus-id) associato a quella query (query-id) e' rilevante per il claim

In teoria, le info di qrels non aggiungon nulla poiche' in origina era pensato per fornire le informazioni che facessero da ponte e collegassero le query e i doc. Nelle versioni piu' moderne di **FEVER** queste info ce le abbiamo nel campo `evidence` delle query, in cui sono registrati gli id dei documenti.

# HOTPOTQA

In [28]:
dataset = 'hotpotqa'
DATA_DIR = Path(f"../data/{dataset}")

In [29]:
corpus, queries, test_queries, train_q, dev_q, test_q = read_all()

In [6]:
print(f"dataset -- {dataset} --")
print(f"Numero documenti del corpus: {len(corpus):,}")
print(f"Numero query:                {len(queries):,}")
print(f"Numero query di test:        {len(test_queries):,}")

dataset -- hotpotqa --
Numero documenti del corpus: 5,233,329
Numero query:                97,852
Numero query di test:        7,405


## Analizziamo i file

In [8]:
queries[20]

{'_id': '5a7f566b5542994857a766f3',
 'text': "Which subject of Michael A. Weinstein's political analyses was a domestic terrorist?",
 'metadata': {'answer': 'Theodore John Kaczynski',
  'supporting_facts': [['Michael A. Weinstein', 0], ['Ted Kaczynski', 0]]}}

Una query ha i seguenti campi:
* `_id`: identificatore univoco
* `text`: e' la domanda vera e propria cui il sistema deve rispondere
* `answer`: e' la risposta corretta alla domanda
* `supporting_facts`: tra i metadata e' il campo piu' importante. Contiene le informazioni necessarie per rispondere alla domanda. In particolare ne abbiamo due, che sta ad indicare che per ricostruire la risposta corretta e' necessario che il sistema estragga almeno due documenti da cui attingere le corrette info. In questo caso sono *Michael A. Weinstein* e *Ted Kaczynski*. Per questa ragione parliamo di **multi hop**: il sistema avra' accesso solo ad info parziali per singolo documento. La risposta corretta necessita del recupero di piu documenti (info parziali) da collettare.<br>Inoltre quei valori nei supporting facts sono proprio i `title` dei doc del corpus cui la query si deve collegare.

In [13]:
corpus[3]

{'_id': '290',
 'title': 'A',
 'text': 'A (named , plural "As", "A\'s", "a"s, "a\'s" or "aes" ) is the first letter and the first vowel of the ISO basic Latin alphabet. It is similar to the Ancient Greek letter alpha, from which it derives. The upper-case version consists of the two slanting sides of a triangle, crossed in the middle by a horizontal bar. The lower-case version can be written in two forms: the double-storey a and single-storey ɑ. The latter is commonly used in handwriting and fonts based on it, especially fonts intended to be read by children, and is also found in italic type.',
 'metadata': {'url': 'https://en.wikipedia.org/wiki?curid=290'}}

Un documento del corpus ha i seguenti campi:
* `_id`: identificatore univoco
* `title`: titolo del documento. E' anche chiave esterna che connette alle query: i titoli dei doc cui la query deve fare riferimento sono nei `supporting_facts`
* `text`: e' un testo sintetico, preso dalla pagina wikipedia associata di cui si disponde dell' *url* tra i metadata

In [15]:
print(next(
    d for d in corpus
    if d["title"] == "Ted Kaczynski"
))

print(next(
    d for d in corpus
    if d["title"] == "Michael A. Weinstein"
))

{'_id': '18621887', 'title': 'Ted Kaczynski', 'text': 'Theodore John Kaczynski ( ; born May 22, 1942), also known as the Unabomber, is an American mathematician, anarchist and domestic terrorist. A mathematical prodigy, he abandoned a promising academic career in 1969, then between 1978 and 1995 killed 3 people, and injured 23 others, in a nationwide bombing campaign that targeted people involved with modern technology. In conjunction, he issued a wide-ranging social critique opposing industrialization and advancing a nature-centered form of anarchism.', 'metadata': {'url': 'https://en.wikipedia.org/wiki?curid=18621887'}}
{'_id': '48897286', 'title': 'Michael A. Weinstein', 'text': 'Michael A. Weinstein (August 24, 1942\xa0– September 17, 2015) was an American political philosopher and political scientist, punk musician, and photography critic. He was a Guggenheim Fellow, Professor of Political Science at Purdue University, and the author or co-author of more than twenty books on a wid

Stampando i due documenti, notiamo come:
* in quello di *Michael A. Weinstein* e' riportato che si e' occupato di Unabomber
* in quello di *Ted Kaczynski* e' riportato che e' anche conosciuto con il nome di Unabomber

Pertanto il sistema ha tutte le info per rispondere alla query.

In [19]:
train_q

,query-id,corpus-id,score
0,5ab6d31155429954757d3384,2921047,1
1,5ab6d31155429954757d3384,158894,1
2,5ac0d92f554299012d1db645,35694141,1
3,5ac0d92f554299012d1db645,12775381,1
4,5abd01335542993a06baf9fc,35216810,1
...,...,...,...
169995,5ae2331e5542992decbdcc4d,263648,1
169996,5a85fda75542996432c57163,6118251,1
169997,5a85fda75542996432c57163,2547979,1
169998,5a7543b155429916b01642cd,15885903,1


E infine ci sono i 3 file di train, val e test che fanno da **ponte** tra query e documenti, riportando le rispettive chiavi primarie.